In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path = "../../../../../../nfs/chengwang_data/ICML_data/result/pancreas/pancreas_res_ini_new.h5ad" 
adata = sc.read_h5ad(file_path)
loadings_df = pd.DataFrame(adata.varm['markov_kernel'], index=adata.var_names)

In [2]:
P = adata.X.copy()
P = P / P.sum(axis=1, keepdims=True)

In [3]:
adata.X = P.copy()

In [ ]:
# index of target factor
target_f = 43
# number of target genes
top_n = 30

# sort and select genes
specific_loadings = loadings_df.iloc[:, target_f].sort_values(ascending=False)
top_genes = specific_loadings.head(top_n)

# visualization
plt.figure(figsize=(10, 8))
sns.barplot(x=top_genes.values, y=top_genes.index, hue=top_genes.index, palette='flare', legend=False)
plt.title(f"Top {top_n} Driver Genes for Batch Factor {target_f} (from markov_kernel)")
plt.xlabel("Markov Kernel Weight")
plt.ylabel("Genes")
plt.grid(axis='x', alpha=0.3)
plt.show()

print(f"--- Factor {target_f} top genes ---")
print(top_genes)

# gene classification
mito = [g for g in top_genes.index if g.upper().startswith('MT-')]
ribo = [g for g in top_genes.index if g.upper().startswith(('RPS', 'RPL'))]
stress = [g for g in top_genes.index if g.upper() in ['FOS', 'JUN', 'DUSP1', 'HSPA1A', 'HSP90AB1']]

if mito: print(f"\n💡 Mitochondrial genes detected: {mito} (may be related to cell viability/death-associated batch effects)")
if ribo: print(f"💡 Ribosomal genes detected: {ribo} (may be related to protein synthesis or technical bias)")
if stress: print(f"💡 Stress-response genes detected: {stress} (may be related to the sample dissociation process)")

In [ ]:
import matplotlib.pyplot as plt

# violin lot
sc.pl.violin(
    adata, 
    keys=['SST'], 
    groupby='tech', 
    rotation=90, 
    show=False 
)

plt.tight_layout()

plt.show()